In [17]:
#!pip install yfinance
import yfinance as yf
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier

def compute_RSI(series, window=14):
    delta = series.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = -delta.where(delta < 0, 0.0)

    avg_gain = gain.rolling(window=window).mean()
    avg_loss = loss.rolling(window=window).mean()

    rs = avg_gain / avg_loss.replace(0,1e-10)
    rsi = 100 - (100 / (1 + rs))
    return rsi

def make_features(ticker):
    data = yf.download("AAPL", start="2020-01-01", end="2025-01-01")
    data['RSI'] = compute_RSI(data['Close'])
    data['Target'] = (data['Close'].shift(-1) > data['Close']).astype(int)
    data['Return'] = data['Close'].pct_change()
    data['SMA_5'] = data['Close'].rolling(window=5).mean()
    data['SMA_10'] = data['Close'].rolling(window=10).mean()

    return data.dropna()

data = make_features("AAPL")
X_train = data[['Return','SMA_5','SMA_10','RSI']]
y_train = data['Target']

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train,y_train)

joblib.dump(model, "stock_model.pkl")
print("Model saved a stock_model.pkl")

C:\Users\modhi\AppData\Local\Temp\ipykernel_17144\2229185052.py:20: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download("AAPL", start="2020-01-01", end="2025-01-01")
[*********************100%***********************]  1 of 1 completed


Model saved a stock_model.pkl
